<a href="https://colab.research.google.com/github/salsabielmesl/flyrank-ml-assignments/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/salsabielmesl/flyrank-ml-assignments/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Type: Classification, Ranking and Scoring.
Rationale: We are predicting whether an individual web page is in a declining performance trend(1 for declining, 0 for growing). Ranking these continuous predictions creates a prioritized queue for content refreshers.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
df = pd.read_csv("../data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(f"Total Dataset Records: {len(df)}")
print("\nTarget Class Breakdown (is_declining_label):")
print(df["is_declining_label"].value_counts(normalize=True))

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target variable: is_declining_label(1 & 0 just as stated above)
Target Def: it's derived from trend_direction feature in the dataset, where the value down represents a page that is losing search visibility and requires content optimization.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
assert "is_declining_label" in df.columns, "Target column missing"
null_count = df["is_declining_label"].isnull().sum()
print(f"missing values in target column: {null_count}")

## 3. Success metric

*One metric you can defend. What number means 'good'?*

primary metric: precision@k and ROC-AUC
Trade of rationale: Content teams have limited editorial bandwidth to rewrite pages each week. Optimizing for Precision@K ensures that among the top K pages flagged for a refresh, the highest possible fraction are actually declining pages, minimizing wasted operational effort.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.metrics import roc_auc_score
y_true = df["is_declining_label"]
dummy_preds = np.full(shpe=len(df), fill_value=0.5)
baseline_auc = roc_auc_score(y_true, dummy_preds)
print(f"Random Guess Baseline ROC-AUC: {baseline_auc:.4f}")

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

unit of Analysis: One row = One unique web page / URL evaluation record.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
key_cols = ["content_age_days", "days_since_last_update", "impressions_90d", "avg_position", "ctr", "word_count", "trend_direction", "is_declining_label"]
df[key_cols].head()

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Why Heuristics Fail: Simple rules (e.g., IF days_since_last_update >= 180 THEN refresh) fail because an old page that maintains high search rankings and high click-through rates does not need a refresh, while a newer page with dropping impressions might.
Why ML Wins: Machine learning models identify non-linear feature interactions (such as high impressions combined with poor position or declining CTR relative to page age), outputting continuous scores that rank urgency far more accurately than rigid thresholds.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_pred"] = stale * visible
rule_accuracy = (df["hand_rule_pred"] == df["is_declining_label"]).mean()
print(f"Simple Heuristic Hand-Rule Accuracy: {rule_accuracy * 100:.2f}%")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.